# Get Study List with UIDs from Production
## Overview
This notebook is used for fetching the list of studies from production to be used for the database comparison notebook ([compare_val_prd.ipynb](compare_val_prd.ipynb)).


## How to use it
1. Set required environment variables in the `.env` file (use the [`.env.example`](.env.example) as a reference).
2. Ensure you have the right SSL certificates on your machine to access the environments.
3. Run the notebook (install any missing dependencies if needed first)
4. The output will be a list of study id's in a text file in the `downloads` folder, named `studylist_with_uid.txt`. 


Output data can be directly copied into the `STUDIES` dictionary in the [compare_val_prd.ipynb](compare_val_prd.ipynb) notebook.

In [ ]:
import ssl
import os
import httpx
import httpx_auth
from datetime import datetime

In [ ]:
%load_ext dotenv
%dotenv
authority = os.environ.get("AUTHORITY_PRD")
scope = os.environ.get("SCOPE_PRD")
url = os.environ.get("URL_PRD")
url2 = os.environ.get("URL_PRD2")
appid = os.environ.get("REG_PRD")

In [ ]:
ctx = ssl.create_default_context(cafile=os.environ.get("REQUESTS_CA_BUNDLE"))

auth = httpx_auth.OAuth2AuthorizationCodePKCE(
    authorization_url=f"https://login.microsoftonline.com/{authority}/oauth2/v2.0/authorize",
    token_url=f"https://login.microsoftonline.com/{authority}/oauth2/v2.0/token",
    client_id=appid,
    scope=scope,
)


# Timeout configuration (10 minutes total, with generous read/connect timeouts)
timeout = httpx.Timeout(
    timeout=600.0,  # total timeout
    connect=30.0,  # connection timeout
    read=300.0,  # read timeout
    write=300.0,  # write timeout
    pool=60.0,  # connection pool timeout
)


client1 = httpx.Client(base_url=url, auth=auth, verify=ctx, timeout=timeout)

print("Test connection: " + str(client1.get("/studies/list?minimal_response=true")))

In [ ]:
def extract_study_info(data_list):
    result = []
    for item in data_list:
        uid = item["uid"]
        id = item["id"]
        result.append({"id": id, "uid": uid})
    return result

def sort_and_display_studies(study_info, output_file="downloads/id_uid_mapping.txt"):
    # Sort studies into groups
    nn_studies = []
    templates_studies = []
    other_studies = []

    for item in study_info:
        id = item["id"]
        if id.startswith("NN"):
            nn_studies.append(item)
        elif id.startswith("TEMPLATES"):
            templates_studies.append(item)
        elif not id.startswith("CDISC"):  # Exclude CDISC studies
            other_studies.append(item)

    # Sort each group by ID
    nn_studies = sorted(nn_studies, key=lambda x: x["id"])
    templates_studies = sorted(templates_studies, key=lambda x: x["id"])
    other_studies = sorted(other_studies, key=lambda x: x["id"])

    # Write to output file
    with open(output_file, 'w') as f:
        # NN studies section
        f.write("# NN Studies\n")
        for item in nn_studies:
            f.write(f'"{item["id"]}": "{item["uid"]}",\n')
        
        f.write("\n# TEMPLATES Studies\n")
        for item in templates_studies:
            f.write(f'"{item["id"]}": "{item["uid"]}",\n')
        
        if other_studies:
            f.write("\n# Other Studies (excluding CDISC)\n")
            for item in other_studies:
                f.write(f'"{item["id"]}": "{item["uid"]}",\n')
    
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"Study information saved to '{output_file}' at {timestamp}")

In [ ]:
# get minimal study list excluding deleted studies
response = client1.get("/studies/list?minimal_response=true&deleted=false")
studies = response.json()
# extract and sort
study_info = extract_study_info(studies)
sort_and_display_studies(study_info)